# SignSage v2 — Kaggle training run (multi-session)

Runs `research/training/train_sign_to_text.py` (corrected post-352868924 config).
Read `research/training/READINESS.md` first. Requires a Kaggle GPU + `HF_TOKEN` secret.
Each step below is one cell; outputs are the run record.

> **V2 EXPERIMENT — NOT PRODUCTION.** A trained v2 checkpoint is an evaluation
> candidate only. Model promotion requires a separate evaluation review against
> the repository's promotion criteria. Nothing in this notebook sets
> `SIGNSAGE_MODEL_PROMOTED` or touches production serving.

**How to run:** do NOT press Run All in the interactive editor for the long run.
Save the notebook, then **Save Version -> Save & Run All** with **GPU enabled**, and let the
batch execution run top-to-bottom. Do not run the training cell manually while the batch job is running.

**Multi-session flow (12h limit):** Session 1 = restore-check -> provision -> preflight ->
audit -> **prepare** (pose cache + tokenizer, resumable) -> overfit gate -> begin training.
Session 2+ = attach prior output as input dataset -> restore cell reuses cache/tokenizer/
checkpoint -> `--resume` continues training. `/kaggle/working` is ephemeral: nothing
persists unless you **Save Version** (persist cell) and re-attach it next session.

Flow: setup -> restore -> provision/preflight -> audit -> **prepare** (`tokenizer,prepare`)
-> **train** (`tokenizer,overfit,train`, overfit gate mandatory) -> evaluate + export ->
persist -> adapter check.



In [ ]:
!pip install -q pose-format jiwer sacrebleu rouge-score sentencepiece huggingface_hub requests tqdm pandas zstandard


In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Private-repo clone: GITHUB_TOKEN via Kaggle Secrets/env, used ONLY for
# this clone through a one-shot HTTP Authorization header. The persisted
# remote stays clean (https://github.com/therandomuser03/signsage.git) and
# the token is never printed, never in history, never in .git/config.
import base64
import json as _json
import os
import subprocess
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/therandomuser03/signsage.git"
DEST = Path("/kaggle/working/signsage")
TRAIN_SCRIPT = "research/training/train_sign_to_text.py"

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
print("GITHUB_TOKEN available:", bool(token))
if not token:
    raise RuntimeError("GITHUB_TOKEN is missing: add it under the Kaggle notebook Add-ons > Secrets, then re-run this cell.")

# Preflight: confirm the token can actually read the private repo.
try:
    _req = urllib.request.Request(
        "https://api.github.com/repos/therandomuser03/signsage",
        headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(_req, timeout=20) as _r:
        _info = _json.loads(_r.read().decode())
except Exception as exc:
    raise RuntimeError("Cannot access private repo therandomuser03/signsage with GITHUB_TOKEN (invalid/expired token or missing repo permission).") from exc
print("private repo accessible:", _info.get("full_name"), "| private:", _info.get("private"))

if not (DEST / ".git").exists():
    _cred = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {_cred}",
                    "clone", REPO_URL, str(DEST)], check=True)
    del _cred
del token

# Post-conditions: clean remote URL, expected training file present.
_remote = subprocess.run(["git", "-C", str(DEST), "remote", "get-url", "origin"],
                         capture_output=True, text=True, check=True).stdout.strip()
assert _remote == REPO_URL, f"remote URL polluted: {_remote!r}"
assert (DEST / TRAIN_SCRIPT).is_file(), f"missing {TRAIN_SCRIPT} in cloned repo"
print("clone OK; remote clean; training script present")

OUT = "/kaggle/working/s2t_v2"
os.makedirs(OUT, exist_ok=True)
print("repo + out ready (run the provisioning cell next)")


In [ ]:
# SESSION RESTORE (run FIRST every session): /kaggle/working is ephemeral.
# If a previous session's output was attached as an input dataset, copy it back
# and report exactly what will be reused. First session: nothing found -> fresh.
# To point at a specific prior output, set PRIOR_OUTPUT_SLUG to its dataset slug
# (the folder name under /kaggle/input) before running this cell.
import json
import os
import shutil
from pathlib import Path

OUT = "/kaggle/working/s2t_v2"
restored = None
_marker = Path(OUT) / "checkpoints" / "signsage_transformer_recovery.pt"
if not _marker.is_file():
    cands = []
    slug = os.environ.get("PRIOR_OUTPUT_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "s2t_v2")
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").glob("*/s2t_v2"))
    for _c in cands:
        if ((_c / "checkpoints").is_dir() or (_c / "tokenizer").is_dir()
                or (_c / "pose_cache").is_dir()):
            shutil.copytree(_c, OUT, dirs_exist_ok=True)
            restored = str(_c)
            break
print("restored from:", restored or "(fresh session -- nothing to reuse yet)")

# Status report: exactly what the later cells will reuse vs rebuild.
_man = Path(OUT) / "pose_cache" / "manifest.json"
if _man.is_file():
    try:
        _m = json.loads(_man.read_text())
        print(f"pose cache: {_m.get('n', '?')} samples, complete={_m.get('complete')}")
    except ValueError:
        print("pose cache: manifest unreadable (will be rebuilt)")
else:
    print("pose cache: ABSENT (prepare cell will build it)")
_tok = Path(OUT) / "tokenizer" / "signsage_en_sp.model"
print("tokenizer:", "REUSE " + str(_tok) if _tok.is_file() else "ABSENT (prepare cell will train it)")
print("token cache:", "REUSE" if (Path(OUT) / "tokenizer" / "tokens.npz").is_file() else "ABSENT")
_rec = Path(OUT) / "checkpoints" / "signsage_transformer_recovery.pt"
if _rec.is_file():
    try:
        import torch
        _r = torch.load(_rec, map_location="cpu", weights_only=False)
        print(f"recovery checkpoint: REUSE (epoch={_r.get('epoch')}, "
              f"best={float(_r.get('best', float('nan'))):.4f}, history={len(_r.get('history', []))})")
    except Exception as exc:
        print(f"recovery checkpoint: present but unreadable ({type(exc).__name__})")
else:
    print("recovery checkpoint: ABSENT (training starts at epoch 1)")
_best = Path(OUT) / "checkpoints" / "signsage_transformer_full.pt"
print("best checkpoint:", "REUSE " + str(_best) if _best.is_file() else "ABSENT")



In [ ]:
# Dataset provisioning: resolve iSign_v1.1.csv from (1) an existing working
# copy, (2) an attached Kaggle dataset (set ISIGN_DATASET_SLUG if attached
# under a custom slug), or (3) a Hugging Face download. Fails early with a
# clear error when none is available -- training never starts without data.
# Also guarantees HF_TOKEN is exported for pose streaming (--pose-source
# remote always needs it): subprocesses inherit os.environ, plain Python
# locals do not. The token itself is never printed.
import os
from pathlib import Path

def _resolve_csv():
    cands = [Path("/kaggle/working/isign/iSign_v1.1.csv")]
    slug = os.environ.get("ISIGN_DATASET_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "iSign_v1.1.csv")
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").glob("*/iSign_v1.1.csv"))
    for _p in cands:
        if _p.is_file():
            return _p
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = os.environ.get("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("HF_TOKEN Kaggle Secret (or env) is required to download iSign from Hugging Face.") from exc
    if not _tok:
        raise RuntimeError("HF_TOKEN is empty: attach the CSV as a Kaggle dataset or set the HF_TOKEN secret.")
    os.environ["HF_TOKEN"] = _tok
    del _tok
    from huggingface_hub import hf_hub_download
    _dest = Path("/kaggle/working/isign")
    _dest.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(repo_id="Exploration-Lab/iSign", filename="iSign_v1.1.csv",
                                repo_type="dataset", token=os.environ["HF_TOKEN"], local_dir=str(_dest)))

CSV_PATH = str(_resolve_csv())
print("iSign CSV:", CSV_PATH)
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient as _USC
        _t = _USC().get_secret("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
        del _t
    except Exception as exc:
        raise RuntimeError("HF_TOKEN is required for remote pose streaming (--pose-source remote): set the HF_TOKEN Kaggle Secret, then re-run.") from exc
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is empty: set the HF_TOKEN Kaggle Secret for remote pose streaming, then re-run.")
print("HF_TOKEN available for pose streaming:", True)


In [ ]:
# Preflight (re-run any time): prove a REAL subprocess inherits HF_TOKEN
# before any expensive command. Reports availability + length ONLY --
# the token value is never printed, written, or embedded anywhere.
import os
import subprocess
import sys

_tok = os.environ.get("HF_TOKEN")
print("HF_TOKEN available:", bool(_tok))
if not _tok:
    raise RuntimeError("HF_TOKEN missing in kernel env: re-run the provisioning cell above (it pulls the Kaggle Secret into os.environ), then re-run this cell.")
_probe = subprocess.run(
    [sys.executable, "-c",
     "import os; _t=os.environ.get('HF_TOKEN'); "
     "print('subprocess HF_TOKEN available:', bool(_t)); "
     "print('token length:', len(_t) if _t else 0)"],
    capture_output=True, text=True)
print(_probe.stdout.strip())
if _probe.returncode != 0 or "subprocess HF_TOKEN available: True" not in _probe.stdout:
    raise RuntimeError("subprocess cannot see HF_TOKEN: restart the kernel, run cells top-to-bottom (provisioning, then this cell), and retry.")
print("preflight OK: training subprocesses will inherit HF_TOKEN")


In [ ]:
# Pose-source preflight (cheap, re-run any time): proves the remote pose
# archive is reachable via HTTP Range reads AND that one REAL pose member
# decodes end-to-end -- before any expensive stage. Writes
# /kaggle/working/s2t_v2/pose_preflight.json and exits; no training starts.
# Requires: provisioning + HF_TOKEN preflight cells above.
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --preflight-only

In [ ]:
!cd /kaggle/working/signsage && python research/training/audit_dataset.py --csv {CSV_PATH} --out /kaggle/working/s2t_v2/dataset_audit.json

In [ ]:
# PHASE A+B PREPARE (Session 1; resumable -- safe to re-run after interruption).
# Materializes pose_cache/ from remote ONCE (sharded ZIPs of verbatim .pose bytes;
# processed fp32 tensors would be ~120 GiB and are deliberately NOT cached) and
# trains the tokenizer + token cache. Re-running skips present samples via the
# manifest and reuses tokenizer artifacts. Requires: restore + provision + preflight.
# NOTE: this needs HF_TOKEN (remote reads) only until the cache is complete.
# MEASURED 2026-10-07: ~1211 KiB/sample -> ~143 GiB full cache at 1.2 samples/s
# sequential (~29h). --jobs 8 fetches in parallel (expect ~4h; watch the ETA
# the cell prints after 200 samples). If the ETA exceeds the session, EITHER
# let prepare span sessions (it resumes) OR add "--subset", "20000" for a pilot
# run (explicit, logged; representation unchanged, just fewer videos).
import subprocess
import sys

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    "--stages", "tokenizer,prepare",
    "--jobs", "8",
    "--resume",
]

print("========================================")
print("PREPARING POSE CACHE + TOKENIZER")
print("(resumable: interrupt any time, re-run this cell)")
print("========================================")

result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
if result.returncode != 0:
    raise RuntimeError("Prepare failed (partial cache is kept -- just re-run this cell).")
print("PREPARE COMPLETE (or already complete -- see 'already complete' above)")




In [ ]:
# MEASURE (dry-run, ~2,000 samples): decides whether the lossless D-cache fits.
# Runs the EXACT V2 preprocessing path, zstd-compresses each fixed 160x1629
# fp32 tensor, verifies byte-exact round-trips, and writes
# /kaggle/working/s2t_v2/measure_report.json with the FIT / BORDERLINE /
# DOES_NOT_FIT verdict. Creates NO cache. Run this BEFORE any full prepare:
# no full training starts until the verdict is reviewed.
# --persistence-target-gib is YOUR measured budget (df -h, trial saves), never
# assumed: omit it and the verdict stays UNKNOWN-no-target.
import subprocess
import sys

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    "--stages", "measure",
    "--measure-samples", "2000",
]

print("========================================")
print("MEASURING ZSTD PROJECTION (dry-run)")
print("========================================")

result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
if result.returncode != 0:
    raise RuntimeError("Measure failed.")

import json as _json
_rep = _json.load(open("/kaggle/working/s2t_v2/measure_report.json"))
print("\ntrue_len: min=%(min)s median=%(median).0f mean=%(mean).1f p90=%(p90).0f p95=%(p95).0f max=%(max)s" % _rep["true_len"])
print(f"compression ratio: {_rep['compression_ratio']:.2f}x "
      f"(dense-data ratio {_rep['dense_data_compression_ratio']:.2f}x, "
      f"zero-padding fraction {_rep['mean_zero_padding_fraction']:.1%})")
print(f"projected full 123,779: {_rep['projected_compressed_gib_full_123779']:.1f} GiB")
print(f"by split: " + ", ".join(f"{k}={v:.1f} GiB" for k, v in _rep["projected_compressed_gib_by_split"].items()))
print(f"verdict: {_rep['verdict']}")



In [ ]:
# V2 TRAINING (reads LOCAL CACHE when prepare is complete; --resume continues).
# Requires: restore + provision + preflight cells above (HF_TOKEN in subprocess env
# for the first session's remote reads; later sessions train purely from disk).
# Stages: tokenizer (reuse), overfit-gate (mandatory, fail-closed, runs on the
# LOCAL cache), train. The gate MUST pass here (64 samples, >=95% exact) before
# any multi-hour spend; a failure aborts, not burns GPU.
#
# RESUME ACROSS SESSIONS: before running this cell in session N+1, attach the
# prior session's saved output as an input dataset and run the restore cell
# (set PRIOR_OUTPUT_SLUG if the slug is ambiguous). Then re-run this cell as-is.

import json as _json
import os
import subprocess
import sys
import time

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    "--epochs", "12",
    "--max-hours", "7",
    "--resume",
    "--stages", "tokenizer,overfit,train",
]

# Split counts come from the audit cell's report, never hardcoded.
_audit_path = "/kaggle/working/s2t_v2/dataset_audit.json"
try:
    _split = _json.load(open(_audit_path))["split"]
    _counts = (f"Train: {_split['train_rows']:,} samples\n"
               f"Validation: {_split['val_rows']:,} samples\n"
               f"Test: {_split['test_rows']:,} samples")
except (OSError, ValueError, KeyError):
    _counts = ("Train/Validation/Test: UNKNOWN "
               "(dataset_audit.json not found -- run the audit cell first)")

print("========================================")
print("STARTING SIGNSAGE V2 TRAINING")
print("========================================")
print(_counts)
print("Epoch budget: 12")
print("Training wall-clock budget: 7 hours")
print("Resume: ENABLED")
print("========================================")

started = time.time()

result = subprocess.run(
    cmd,
    cwd="/kaggle/working/signsage"
)

elapsed = (time.time() - started) / 3600

print(f"\nTraining process exited with code {result.returncode}")
print(f"Training runtime: {elapsed:.2f} hours")

if result.returncode != 0:
    raise RuntimeError(
        "V2 training failed. Do not proceed to deployment/export."
    )

print("V2 TRAINING COMPLETED SUCCESSFULLY")



In [ ]:
# Final test evaluation + deployment-bundle export (reads LOCAL CACHE).
# Requires: restore + provision cells above.
import json
import os
import subprocess
import sys

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    "--stages", "evaluate,export",
]

print("========================================")
print("RUNNING FINAL TEST + EXPORT")
print("========================================")

result = subprocess.run(
    cmd,
    cwd="/kaggle/working/signsage"
)

if result.returncode != 0:
    raise RuntimeError(
        "Evaluation/export failed."
    )

report_path = "/kaggle/working/s2t_v2/evaluation_report.json"

if not os.path.isfile(report_path):
    raise FileNotFoundError(report_path)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("\nTEST METRICS:")
print(report.get("test_metrics"))

print("\nDIVERSITY:")
print(report.get("diversity"))

print("\nQUALITY GATE:")
print(report.get("gate"))

print("\nEvaluation report:")
print(report_path)

print("\nDeployment bundle:")
print("/kaggle/working/s2t_v2/signsage_deployment_bundle")



In [ ]:
# PERSIST (run LAST every session): verify what Save Version will keep, then
# use Kaggle's Save Version button to snapshot /kaggle/working as an output
# dataset. Next session: attach that output as input + set PRIOR_OUTPUT_SLUG
# (or rely on auto-detect in the restore cell). Copying the .ipynb alone
# transfers NOTHING -- only a saved output version persists the cache.
# Secrets check: HF_TOKEN/GITHUB_TOKEN must never appear in the output dir.
import json
import os
from pathlib import Path
# PERSISTENCE DIAGNOSTICS (cheap, run any time): ephemeral-disk usage plus the
# labelled size of the current output/cache. No Kaggle storage limit is assumed
# or hardcoded anywhere: compare these numbers against YOUR measured budget
# (df output below + trial Save Version sizes) before relying on persistence.
import subprocess as _sp
_r = _sp.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("--- df -h /kaggle/working ---")
print(_r.stdout.strip() or "(df unavailable)")
print("--- current output/cache sizes (persistence check, NOT a limit) ---")

OUT = Path("/kaggle/working/s2t_v2")
total = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
print(f"output size: {total / 2**30:.2f} GiB in {OUT}")
for _sub in ("pose_cache", "tokenizer", "checkpoints"):
    _d = OUT / _sub
    if _d.is_dir():
        _n = sum(1 for _ in _d.rglob("*") if _.is_file())
        _s = sum(p.stat().st_size for p in _d.rglob("*") if p.is_file()) / 2**30
        print(f"  {_sub}: {_n} files, {_s:.2f} GiB")
_man = OUT / "pose_cache" / "manifest.json"
if _man.is_file():
    try:
        _m = json.loads(_man.read_text())
        print(f"  pose cache complete={_m.get('complete')} n={_m.get('n')}")
    except ValueError:
        print("  pose cache manifest UNREADABLE")
for _name in ("HF_TOKEN", "GITHUB_TOKEN"):
    _hits = [str(p) for p in OUT.rglob("*") if p.is_file()
             and _name in p.name.upper()]
    assert not _hits, f"secret-named file in output: {_hits}"
print("no secret-named files in output dir")
print()
print("NEXT: Kaggle Save Version (save output), then in a new session attach")
print("that output as input data and run the restore cell (PRIOR_OUTPUT_SLUG")
print("only if auto-detect finds the wrong s2t_v2).")




In [ ]:
# Verify the exported bundle loads through the REAL serving adapter (no notebook state).
import os
import sys
import numpy as np

sys.path.insert(0, "/kaggle/working/signsage/backend")

from app.ml.adapters.sign_to_text_adapter import SignToTextAdapter

bundle = "/kaggle/working/s2t_v2/signsage_deployment_bundle"

if not os.path.isdir(bundle):
    raise FileNotFoundError(bundle)

adapter = SignToTextAdapter.load(
    bundle,
    device="cpu"
)

frames = np.random.default_rng(0).normal(
    size=(25, 543, 3)
).astype(np.float32)

result = adapter.transcribe(frames)

print(
    "bundle loads + transcribes:",
    result.accepted,
    result.reason,
    repr(result.text[:80])
)

print(
    "IMPORTANT: random-input output does NOT measure translation quality."
)
print(
    "Use evaluation_report.json for model quality."
)